# Random Projection (RP)

Notebook นี้เปรียบเทียบ Gaussian และ Sparse Random Projection บน Digits dataset โดยประเมินการบิดเบือนของระยะทาง ความแม่นยำ เวลา และหน่วยความจำ

In [ ]:
from time import perf_counter

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from sklearn.datasets import load_digits
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.random_projection import GaussianRandomProjection, SparseRandomProjection

RANDOM_STATE = 42
TARGET_DIMENSIONS = [16, 32, 48, 64]
PAIR_SAMPLE_SIZE = 20_000

## 1. Import Libraries and Load Dataset

Digits มีภาพตัวเลขขนาด $8 \times 8$ จึงมี 64 features และ 10 classes.

In [ ]:
digits = load_digits()
X, y = digits.data, digits.target

print('Feature matrix shape:', X.shape)
print('Number of classes:', len(np.unique(y)))
pd.Series(y, name='digit').value_counts().sort_index().to_frame('count')

In [ ]:
X

In [ ]:
y

## 2. Prepare and Standardize Features

แยก test set ก่อน จากนั้น fit `StandardScaler` เฉพาะ training set เพื่อไม่ให้ข้อมูลจาก test set รั่วเข้าสู่ขั้นตอน preprocessing.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.25,
    stratify=y,
    random_state=RANDOM_STATE,
)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print('Training shape:', X_train_scaled.shape)
print('Test shape:', X_test_scaled.shape)

## 3. Apply Gaussian Random Projection

สำหรับข้อมูลแบบแถว ใช้ $X_{\text{reduced}} = XR$ โดย Gaussian RP สุ่มสมาชิกของ $R$ จากการแจกแจงปกติ และลด 64 features เหลือ $k$ features.

In [ ]:
gaussian_representations = {}
gaussian_metrics = []

for target_dimension in TARGET_DIMENSIONS:
    projector = GaussianRandomProjection(
        n_components=target_dimension,
        random_state=RANDOM_STATE,
    )
    start_time = perf_counter()
    X_train_projected = projector.fit_transform(X_train_scaled)
    X_test_projected = projector.transform(X_test_scaled)
    transform_seconds = perf_counter() - start_time

    gaussian_representations[target_dimension] = (X_train_projected, X_test_projected)
    gaussian_metrics.append({
        'method': 'Gaussian RP',
        'k': target_dimension,
        'transform_seconds': transform_seconds,
        'train_shape': X_train_projected.shape,
        'test_shape': X_test_projected.shape,
    })

pd.DataFrame(gaussian_metrics)

## 4. Apply Sparse Random Projection

Sparse RP ใช้เมทริกซ์ $R$ ที่ส่วนใหญ่เป็นศูนย์ จึงเหมาะเมื่อ input มีความ sparse หรือเมื่อต้องการลดต้นทุนการคูณเมทริกซ์.

In [ ]:
sparse_representations = {}
sparse_metrics = []

for target_dimension in TARGET_DIMENSIONS:
    projector = SparseRandomProjection(
        n_components=target_dimension,
        random_state=RANDOM_STATE,
    )
    start_time = perf_counter()
    X_train_projected = projector.fit_transform(X_train_scaled)
    X_test_projected = projector.transform(X_test_scaled)
    transform_seconds = perf_counter() - start_time

    sparse_representations[target_dimension] = (X_train_projected, X_test_projected)
    sparse_metrics.append({
        'method': 'Sparse RP',
        'k': target_dimension,
        'transform_seconds': transform_seconds,
        'train_shape': X_train_projected.shape,
        'test_shape': X_test_projected.shape,
    })

pd.DataFrame(sparse_metrics)

## 5. Measure Pairwise Distance Distortion

สุ่มคู่ข้อมูลที่ต่างกัน แล้วคำนวณ

$$\frac{|d_{\mathrm{projected}} - d_{\mathrm{original}}|}{d_{\mathrm{original}}}$$

โดยรายงาน mean, median, percentile 95 และ maximum เพื่อไม่ให้ค่าเฉลี่ยซ่อนคู่ที่เพี้ยนมาก.

In [ ]:
def distance_distortion_summary(original_data, projected_data, sample_size, random_state):
    """Return relative-distance-distortion summaries from random distinct pairs."""
    generator = np.random.default_rng(random_state)
    first_indices = generator.integers(0, len(original_data), size=sample_size)
    second_indices = generator.integers(0, len(original_data), size=sample_size)

    same_pair = first_indices == second_indices
    while same_pair.any():
        second_indices[same_pair] = generator.integers(
            0,
            len(original_data),
            size=same_pair.sum(),
        )
        same_pair = first_indices == second_indices

    original_distances = np.linalg.norm(
        original_data[first_indices] - original_data[second_indices],
        axis=1,
    )
    projected_distances = np.linalg.norm(
        projected_data[first_indices] - projected_data[second_indices],
        axis=1,
    )
    nonzero_distance = original_distances > 0
    distortion = np.abs(projected_distances[nonzero_distance] - original_distances[nonzero_distance]) / original_distances[nonzero_distance]

    return {
        'mean_distortion': distortion.mean(),
        'median_distortion': np.median(distortion),
        'p95_distortion': np.percentile(distortion, 95),
        'max_distortion': distortion.max(),
    }

In [ ]:
distortion_rows = []

for method_name, representations in {
    'Gaussian RP': gaussian_representations,
    'Sparse RP': sparse_representations,
}.items():
    for target_dimension, (X_train_projected, _) in representations.items():
        summary = distance_distortion_summary(
            X_train_scaled,
            X_train_projected,
            PAIR_SAMPLE_SIZE,
            RANDOM_STATE,
        )
        distortion_rows.append({'method': method_name, 'k': target_dimension, **summary})

distortion_results = pd.DataFrame(distortion_rows)
distortion_results.round(3)

## 6. Compare Downstream Classification Performance

ใช้ Logistic Regression ตัวเดิมกับ features เดิมและ features หลัง RP แล้ววัด accuracy บน test set เดียวกัน.

In [ ]:
def evaluate_classifier(X_train_features, X_test_features, y_train, y_test):
    classifier = LogisticRegression(max_iter=2_000, random_state=RANDOM_STATE)
    start_time = perf_counter()
    classifier.fit(X_train_features, y_train)
    training_seconds = perf_counter() - start_time
    test_accuracy = accuracy_score(y_test, classifier.predict(X_test_features))
    return test_accuracy, training_seconds

accuracy_rows = []
baseline_accuracy, baseline_training_seconds = evaluate_classifier(
    X_train_scaled,
    X_test_scaled,
    y_train,
    y_test,
)
accuracy_rows.append({
    'method': 'Original features',
    'k': X_train_scaled.shape[1],
    'test_accuracy': baseline_accuracy,
    'classifier_training_seconds': baseline_training_seconds,
})

for method_name, representations in {
    'Gaussian RP': gaussian_representations,
    'Sparse RP': sparse_representations,
}.items():
    for target_dimension, (X_train_projected, X_test_projected) in representations.items():
        test_accuracy, training_seconds = evaluate_classifier(
            X_train_projected,
            X_test_projected,
            y_train,
            y_test,
        )
        accuracy_rows.append({
            'method': method_name,
            'k': target_dimension,
            'test_accuracy': test_accuracy,
            'classifier_training_seconds': training_seconds,
        })

accuracy_results = pd.DataFrame(accuracy_rows)
accuracy_results.round(3)

## 7. Compare Runtime and Memory Use

เวลาและหน่วยความจำเป็นส่วนหนึ่งของการเลือก $k$ โดยคำนวณขนาด representation แบบประมาณจาก array ที่เก็บในหน่วยความจำ.

In [ ]:
def representation_memory_mb(features):
    if hasattr(features, 'data') and not isinstance(features, np.ndarray):
        byte_count = features.data.nbytes + features.indices.nbytes + features.indptr.nbytes
    else:
        byte_count = features.nbytes
    return byte_count / 1_000_000

runtime_results = pd.concat(
    [pd.DataFrame(gaussian_metrics), pd.DataFrame(sparse_metrics)],
    ignore_index=True,
)
runtime_results['representation_memory_mb'] = [
    representation_memory_mb(
        (gaussian_representations if method == 'Gaussian RP' else sparse_representations)[target_dimension][0]
    )
    for method, target_dimension in zip(runtime_results['method'], runtime_results['k'])
]
comparison_results = (
    distortion_results
    .merge(accuracy_results, on=['method', 'k'])
    .merge(runtime_results.drop(columns=['train_shape', 'test_shape']), on=['method', 'k'])
)
comparison_results.round(3)

## 8. Visualize Projection Quality Across Dimensions

เปรียบเทียบ trade-off: โดยทั่วไป $k$ มากขึ้นจะลด distortion แต่ใช้เวลาและหน่วยความจำมากขึ้น.

In [ ]:
figure, axes = plt.subplots(2, 2, figsize=(12, 8), sharex=True)

for method_name, method_results in comparison_results.groupby('method'):
    method_results = method_results.sort_values('k')
    axes[0, 0].plot(method_results['k'], method_results['mean_distortion'], marker='o', label=method_name)
    axes[0, 1].plot(method_results['k'], method_results['test_accuracy'], marker='o', label=method_name)
    axes[1, 0].plot(method_results['k'], method_results['transform_seconds'], marker='o', label=method_name)
    axes[1, 1].plot(method_results['k'], method_results['representation_memory_mb'], marker='o', label=method_name)

axes[0, 0].set(title='Mean distance distortion', ylabel='relative distortion')
axes[0, 1].set(title='Test accuracy', ylabel='accuracy')
axes[1, 0].set(title='Projection time', xlabel='target dimension k', ylabel='seconds')
axes[1, 1].set(title='Training representation memory', xlabel='target dimension k', ylabel='MB')

for axis in axes.flat:
    axis.grid(alpha=0.25)
    axis.legend(frameon=False)

plt.tight_layout()
plt.show()

## Interpretation Checklist

- เลือก $k$ ที่ distortion อยู่ในระดับยอมรับได้สำหรับโจทย์
- ยืนยันว่า accuracy ไม่ลดลงมากเมื่อเทียบกับ original features
- เปรียบเทียบ Gaussian กับ Sparse RP ภายใต้เวลาและ memory budget เดียวกัน
- ทดลองหลาย `random_state` หากต้องการวัดความเสถียรของ random projection
- สำหรับข้อมูลขนาดใหญ่ ให้สุ่มคู่ข้อมูลเพื่อวัด distortion แทนการคำนวณทุกคู่